In [1]:
# ======================================================
# Part 1: 环境准备
# ======================================================
%pip install -U google-generativeai numpy scikit-learn

In [2]:
# ======================================================
# Part 2: 配置 API 密钥
# ======================================================
import os
import google.generativeai as genai
import getpass

api_key = getpass.getpass('请输入 Gemini API 密钥: ')
genai.configure(api_key=api_key)
print("成功配置 API 密钥！")

请输入 Gemini API 密钥: ··········
成功配置 API 密钥！


In [3]:
# ======================================================
# Part 3: 创建并加载知识库
# ======================================================
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# --- 1. 创建本地知识库文件 ---
knowledge_base_text = """
Happy-LLM 是一个由 Datawhale 开源社区精心打造的、从零开始的大语言模型原理与实践教程。这个项目的核心使命是降低大模型（LLM）的学习门槛，为广大AI爱好者和初学者提供一条清晰、系统、由浅入深的进阶路径。它不仅详细剖析了 LLM 的核心理论，还强调动手实践，旨在实现“授之以渔”。

教程的内容结构遵循了 NLP 领域的技术发展脉络。第一章到第四章是理论基础部分。第一章从 NLP 的基本概念和发展历程讲起；第二章深入讲解了作为所有现代 LLM 基石的 Transformer 架构，包括其核心组件——注意力机制；第三章则梳理了经典的预训练语言模型（PLM）三大流派：Encoder-Only (BERT), Encoder-Decoder (T5), 和 Decoder-Only (GPT)，为理解 LLM 的架构演进打下坚实基础。第四章则正式进入大语言模型的世界，详细阐述了 LLM 的定义、涌现能力、训练三阶段（Pretrain, SFT, RLHF）等核心概念。

第五章是整个教程的第一个动手实践章节，其主题是“动手实现一个 LLaMA2 大模型”。在这一章，读者将使用 PyTorch，像搭积木一样，从零开始亲手构建 LLaMA2 的每一个核心组件，包括 RMSNorm、旋转位置编码（RoPE）、分组查询注意力（GQA）以及 SwiGLU 前馈网络。最终，所有组件会被组装成一个完整的、可运行的 Transformer 模型。这一章的目标是让学习者彻底告别“黑盒”，深入理解模型内部的每一个细节。

第六章则从“手搓”转向工业界标准，介绍了如何使用 Hugging Face 的 `transformers` 和 `peft` 库，来高效地进行模型训练和微调。重点讲解了 `Trainer` 的使用方法，以及如何通过 LoRA 这种参数高效微调（Parameter-Efficient Fine-Tuning）技术，在有限的资源下对大模型进行定制化训练。这一章的目标是让学习者掌握在实际工作中进行模型微调的主流方法。

第七章聚焦于 LLM 的高级应用，是衔接学术与产业的重要桥梁。本章详细介绍了两大前沿技术：检索增强生成（Retrieval-Augmented Generation, RAG）和智能体（Agent）。RAG 部分解释了如何通过外挂知识库来解决 LLM 的幻觉和知识过时问题，并提供了一个搭建简单 RAG 系统的实例。Agent 部分则介绍了如何通过工具调用（Tool Calling）等技术，赋予 LLM “行动”的能力，使其能够与外部世界交互并完成复杂任务。
"""
file_path = "knowledge_base.txt"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(knowledge_base_text.strip())

print(f"知识库文件 '{file_path}' 创建成功！")


# --- 2. 定义文档加载与切分函数 ---
def load_and_chunk_knowledge_base(file_path, chunk_size=200):
    """读取文档并将其切分成小块"""
    print(f"\n--- 正在从 '{file_path}' 加载知识库 ---")
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    print(f"--- 文档被切分成 {len(chunks)} 个块 ---")
    return chunks

# --- 3. 执行加载和切分 ---
knowledge_chunks = load_and_chunk_knowledge_base(file_path)

# 打印一个块看看效果
print("\n文档块示例:")
print(repr(knowledge_chunks[0]))

知识库文件 'knowledge_base.txt' 创建成功！

--- 正在从 'knowledge_base.txt' 加载知识库 ---
--- 文档被切分成 6 个块 ---

文档块示例:
'Happy-LLM 是一个由 Datawhale 开源社区精心打造的、从零开始的大语言模型原理与实践教程。这个项目的核心使命是降低大模型（LLM）的学习门槛，为广大AI爱好者和初学者提供一条清晰、系统、由浅入深的进阶路径。它不仅详细剖析了 LLM 的核心理论，还强调动手实践，旨在实现“授之以渔”。\n\n教程的内容结构遵循了 NLP 领域的技术发展脉络。第一章到第四章是理论基础部分。第一章从 NLP '


In [4]:
# ======================================================
# Part 4: 搭建 RAG 核心组件
# ======================================================

# --- 1. 定义向量化函数 ---
def get_embeddings(texts, model_name="gemini-embedding-001", task_type="retrieval_document"):
    """使用 Gemini Embedding 模型将文本块向量化"""
    print(f"--- 正在使用 '{model_name}' 进行向量化... ---")
    return genai.embed_content(
        model=model_name,
        content=texts,
        task_type=task_type
    )['embedding']

# --- 2. 定义检索函数 ---
def retrieve_top_k_chunks(query_embedding, doc_embeddings, docs, k=1):
    """根据查询向量，检索最相似的 k 个文档块"""
    print("--- 正在进行向量检索... ---")
    sims = cosine_similarity([query_embedding], doc_embeddings)[0]
    top_k_indices = np.argsort(sims)[-k:][::-1]
    return [docs[i] for i in top_k_indices]

# --- 3. 定义 RAG 函数 ---
def generate_answer_with_rag(query, context, model_name="gemini-2.5-flash-lite"):
    """使用检索到的上下文来增强 LLM 的生成"""
    prompt = f"""
    请根据以下上下文信息，来回答用户的问题。
    如果你在上下文中找不到答案，就说你不知道。

    上下文:
    ---
    {context}
    ---

    问题: {query}
    """
    model = genai.GenerativeModel(model_name)
    response = model.generate_content(prompt)
    return response.text

In [7]:
# ======================================================
# Part 5: RAG vs. Non-RAG
# ======================================================

# --- 1. 对所有知识库块进行向量化 (RAG 准备工作) ---
document_embeddings = get_embeddings(knowledge_chunks)
print(f"--- 成功生成 {len(document_embeddings)} 个文档向量 ---\n")

--- 正在使用 'gemini-embedding-001' 进行向量化... ---
--- 成功生成 6 个文档向量 ---



In [6]:
# --- 2. 提出你的问题 ---
user_query = "Happy-LLM 教程的第五章和第七章主要讲了什么内容？"
print("="*60)
print(f"用户问题: {user_query}")
print("="*60)


def generate_answer_without_rag(query, model_name="gemini-2.5-flash-lite"):
    """直接调用 LLM 生成答案"""
    # 简单的 Prompt，不提供任何上下文
    prompt = f"请回答以下问题: {query}"

    model = genai.GenerativeModel(model_name)
    response = model.generate_content(prompt)
    return response.text

# --- 获取 Non-RAG 的回答 ---
answer_without_rag = generate_answer_without_rag(user_query)

# --- 向量化问题 ---
query_embedding = get_embeddings([user_query], task_type="retrieval_query")[0]

# --- 检索最相关的文档块 ---
retrieved_chunks = retrieve_top_k_chunks(query_embedding, document_embeddings, knowledge_chunks, k=2)
context_str = "\n".join(retrieved_chunks)
print(f"\n--- 检索到的最相关上下文:\n{context_str}\n---")

# --- 使用检索到的上下文生成最终答案 ---
answer_with_rag = generate_answer_with_rag(user_query, context_str)


# ======================================================
# 最终结果对比
# ======================================================
print("\n" + "#"*60)
print("结果对比")
print("#"*60)

print("\n【不使用 RAG 的回答】:")
print("--------------------")
print(answer_without_rag)

print("\n【使用 RAG 的回答】:")
print("--------------------")
print(answer_with_rag)

用户问题: Happy-LLM 教程的第五章和第七章主要讲了什么内容？
--- 正在使用 'gemini-embedding-001' 进行向量化... ---
--- 正在进行向量检索... ---

--- 检索到的最相关上下文:
Efficient Fine-Tuning）技术，在有限的资源下对大模型进行定制化训练。这一章的目标是让学习者掌握在实际工作中进行模型微调的主流方法。

第七章聚焦于 LLM 的高级应用，是衔接学术与产业的重要桥梁。本章详细介绍了两大前沿技术：检索增强生成（Retrieval-Augmented Generation, RAG）和智能体（Agent）。RAG 部分解释了如何通过外挂知识库来解决 L
LLM 的定义、涌现能力、训练三阶段（Pretrain, SFT, RLHF）等核心概念。

第五章是整个教程的第一个动手实践章节，其主题是“动手实现一个 LLaMA2 大模型”。在这一章，读者将使用 PyTorch，像搭积木一样，从零开始亲手构建 LLaMA2 的每一个核心组件，包括 RMSNorm、旋转位置编码（RoPE）、分组查询注意力（GQA）以及 SwiGLU 前馈网络。最终，所有组件会
---

############################################################
结果对比
############################################################

【不使用 RAG 的回答】:
--------------------
关于“Happy-LLM”这个教程，我需要**澄清一个信息**：**目前我无法直接找到一个名为“Happy-LLM”的官方、公开且广为流传的教程。**

因此，我无法确切地告诉你它的第五章和第七章讲了什么。

**为什么会出现这种情况？**

*   **教程名称不准确：** 可能教程的实际名称略有不同，或者您记错了。
*   **非公开教程：** “Happy-LLM”可能是一个内部教程、私有课程，或者是一个非常小众的、没有广泛公开的资源。
*   **新近发布的教程：** 如果这是一个非常新的教程，我可能还没有足够的时间将其纳入我的知识库。
*   **特定框架或库的教程：** “Happy-LLM”可能指的是